# 🗺️ Segmentação Baseada em Regiões — SLIC + RAG + Cálculo de Área de Reparo

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Segmentação Avançada e Metrologia

---

## Contexto

No módulo anterior (Otsu) segmentamos trincas pixel a pixel com um único limiar global. Isso funciona bem quando a imagem tem **histograma bimodal claro**. Mas em fotos de pavimentos com reparos, a situação é mais complexa:

- O reparo tem cor e textura **diferentes do pavimento original** — mas não há um único valor de cinza que os separe
- A superfície apresenta variação contínua de brilho (sol, sombra, óleo)
- O objetivo não é apenas detectar a região, mas **medir sua área em metros quadrados** para orçamento e laudo técnico

Para isso, usamos uma abordagem **baseada em regiões**, que trabalha com grupos de pixels semelhantes (superpixels) em vez de pixels individuais.

---

## Conceitos fundamentais

### Superpixels — SLIC

**SLIC** (*Simple Linear Iterative Clustering*) agrupa pixels vizinhos que são simultaneamente:
- **Similares em cor** (espaço LAB — perceptualmente uniforme)
- **Próximos no espaço** (distância euclidiana x,y)

A função de distância combinada é:

$$D = \sqrt{\left(\frac{d_{cor}}{m}\right)^2 + \left(\frac{d_{xy}}{S}\right)^2}$$

onde $m$ é o parâmetro `compactness` e $S$ é o espaçamento entre centros dos superpixels ($S = \sqrt{N/k}$, com $k$ superpixels em $N$ pixels).

| Parâmetro | Efeito de aumentar |
|-----------|-------------------|
| `n_segments` | Mais superpixels, menores e mais detalhados |
| `compactness` | Superpixels mais regulares (quadrados), menos sensíveis à cor |
| `sigma` | Mais suavização pré-SLIC → superpixels mais homogêneos |

### RAG — Region Adjacency Graph

Após o SLIC, cada superpixel vira um **nó do grafo**. Uma aresta conecta dois nós se os superpixels são adjacentes na imagem. O peso da aresta é inversamente proporcional à diferença de cor média entre eles (`mode='similarity'`): superpixels parecidos têm peso alto.

```
Imagem → SLIC → [superpixel 1] ──── [superpixel 2] ──── [superpixel 3]
                     │                   │ (aresta = similaridade de cor)
                 [superpixel 4] ──── [superpixel 5]
```

O RAG permite **mesclar superpixels** com base na similaridade, reduzindo regiões fragmentadas em regiões maiores e coerentes — ideal para delimitar um reparo asfáltico que tem cor uniforme mas bordas irregulares.

### Calibração métrica — escala px → m

Para converter área em pixels para metros quadrados, precisamos de uma **referência de escala** visível na imagem (faixa de sinalização, cone, marco de 10 m):

$$\text{escala} = \frac{L_{ref\ [m]}}{L_{ref\ [px]}} \quad [m/px]$$

$$\text{Área}_{[m^2]} = \text{Área}_{[px^2]} \times \text{escala}^2$$

Este é o mesmo princípio usado em levantamentos topográficos com foto aérea e em BIM (relação modelo-real via IFC QuantitySet).

> ⚠️ **Atenção:** a calibração só é válida se a câmera estiver **perpendicular à superfície** (foto nadir) e a superfície for plana. Fotos oblíquas introduzem distorção de perspectiva que invalida o cálculo direto.

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Instalar dependências e testar com imagem sintética |
| 2 | Montar o Google Drive e configurar parâmetros |
| 3 | Definir o pipeline SLIC + RAG + calibração |
| 4 | Processar imagens em lote |
| 5 | Visualizar superpixels, grafo e área calculada |
| 6 | Calibrar parâmetros SLIC com grade visual |

> 💡 **Recomendação:** execute a Seção 1 primeiro — ela mostra o SLIC funcionando em uma imagem sintética de pavimento sem precisar do Drive.

---
## Seção 1 — Instalar dependências e testar com imagem sintética

### 1a. Instalação

`scikit-image` já vem instalado no Colab. Verificamos apenas a versão.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import skimage
from skimage.segmentation import slic, mark_boundaries
from skimage import graph
from skimage.color import rgb2gray
from skimage.future import graph as future_graph
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd

print(f'OpenCV       : {cv2.__version__}')
print(f'scikit-image : {skimage.__version__}')
print(f'NumPy        : {np.__version__}')

### 1b. Teste com imagem sintética de pavimento

Criamos uma imagem que simula uma foto aérea (nadir) de pista com reparo:

- **Cinza escuro** → asfalto original envelhecido
- **Cinza mais claro e uniforme** → remendo (reparo) mais novo
- **Marcas brancas** → faixa de sinalização (referência de escala)
- **Ruído de textura** → variação real de superfície

**O que esperar:** o SLIC deve agrupar o reparo em poucos superpixels de cor similar, distintos dos superpixels do asfalto original.

In [ ]:
# ── Criar imagem sintética 500×700 (RGB) — pista com reparo
h, w = 500, 700
img_sint = np.full((h, w, 3), (55, 55, 55), dtype=np.uint8)   # asfalto escuro

# Ruído de textura (agregados)
ruido = np.random.normal(0, 14, img_sint.shape).astype(np.int16)
img_sint = np.clip(img_sint.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# Reparo central (mais claro e menos ruidoso — asfalto novo)
reparos = [(slice(150, 360), slice(180, 530))]   # (y, x)
for sy, sx in reparos:
    bloco = np.full((sy.stop-sy.start, sx.stop-sx.start, 3), (100, 100, 100), dtype=np.uint8)
    ruido_rep = np.random.normal(0, 6, bloco.shape).astype(np.int16)
    bloco = np.clip(bloco.astype(np.int16) + ruido_rep, 0, 255).astype(np.uint8)
    img_sint[sy, sx] = bloco

# Faixa de sinalização (referência de escala — 10 m simulados)
cv2.rectangle(img_sint, (20, 20), (120, 50), (230, 230, 230), -1)   # placa de referência
cv2.putText(img_sint, '10 M', (25, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (30, 30, 30), 2)

# Linha de faixa (referência de escala em pixels)
cv2.line(img_sint, (20, 460), (620, 460), (200, 200, 200), 4)   # 600 px = 10 m neste exemplo

# ── Aplicar SLIC
segments = slic(img_sint, n_segments=300, compactness=10, sigma=1, start_label=1)

# ── Construir RAG
rag = graph.rag_mean_color(img_sint, segments, mode='similarity')

# ── Máscara do reparo (referência manual para este teste)
mascara_sint = np.zeros((h, w), dtype=np.uint8)
for sy, sx in reparos:
    mascara_sint[sy, sx] = 255

# ── Calibração: 600 px = 10 m → escala
REF_M  = 10.0
REF_PX = 600.0
escala = REF_M / REF_PX          # m/px
area_px = int(mascara_sint.sum() // 255)
area_m2 = area_px * (escala ** 2)

# ── Visualizar
img_boundaries = mark_boundaries(img_sint, segments, color=(1, 0.8, 0))

fig, eixos = plt.subplots(1, 4, figsize=(24, 5))
dados = [
    (img_sint,        'Original sintético\n(pista + reparo + faixa)', None),
    (img_boundaries,  f'Superpixels SLIC\n(n_segments=300, compactness=10)', None),
    (mascara_sint,    'Máscara do reparo\n(referência manual)', 'gray'),
    (mark_boundaries(img_sint / 255.0,
                     (segments * (mascara_sint > 0)),
                     color=(1, 0.2, 0.2)),
                     f'Superpixels sobre reparo\nÁrea ≈ {area_m2:.1f} m²', None),
]
for ax, (im, t, cmap) in zip(eixos, dados):
    if cmap:
        ax.imshow(im, cmap=cmap)
    else:
        ax.imshow(im if im.dtype == np.float64 or im.max() <= 1.0
                  else im)
    ax.set_title(t, fontsize=9)
    ax.axis('off')

plt.suptitle(
    f'✅ Teste sintético — Superpixels SLIC + RAG + Calibração\n'
    f'Escala: {REF_PX:.0f} px = {REF_M} m  →  1 px = {escala:.4f} m  →  Área reparo ≈ {area_m2:.2f} m²',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Superpixels gerados  : {segments.max()}')
print(f'Escala               : 1 px = {escala:.5f} m  →  1 px² = {escala**2:.6f} m²')
print(f'Área do reparo       : {area_px} px²  →  {area_m2:.2f} m²')
print('\n✅ Ambiente OK — pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar

> ⚠️ **Requisito de calibração:** cada imagem precisa de uma referência de escala visível (faixa, cone, régua, marco viário). Anote o `REF_PX` medindo manualmente com uma ferramenta de imagem (ex: GIMP, Fiji/ImageJ, ou o próprio Label Studio) o comprimento da referência em pixels.

> 📐 **Foto nadir obrigatória:** o cálculo de área só é válido para fotos tiradas **perpendicularmente à superfície**. Fotos oblíquas distorcem a escala — use drone em modo nadir ou câmera em tripé vertical.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
# ── Caminhos
PASTA_ENTRADA = Path('/content/drive/MyDrive/fotos_obra')
PASTA_SAIDA   = Path('/content/drive/MyDrive/fotos_obra_regioes')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Calibração de escala
# REF_M  : comprimento real da referência visível na imagem (metros)
# REF_PX : comprimento da mesma referência medido em pixels na imagem
#           → medir com GIMP, ImageJ ou Label Studio antes de rodar
REF_M  = 10.0    # ex: faixa de 10 m na pista
REF_PX = 500.0   # ex: essa faixa mede 500 px na foto  ← SUBSTITUA PELO VALOR REAL

# ── Parâmetros SLIC
N_SEGMENTS  = 500    # número aproximado de superpixels
COMPACTNESS = 10     # regularidade dos superpixels (5–20)
SIGMA       = 1      # suavização pré-SLIC (0–3)

# ── Região do reparo (fração da imagem) — SUBSTITUA pela lógica real
# Estas frações definem um retângulo central como área de reparo para demonstração.
# Em produção: use máscara de Label Studio, limiar de cor ou clique do usuário.
REPARO_Y0 = 0.30    # y inicial (fração da altura)
REPARO_Y1 = 0.70    # y final
REPARO_X0 = 0.20    # x inicial (fração da largura)
REPARO_X1 = 0.80    # x final

# ── Redimensionamento
LARGURA_MAX = 1200

# ── Extensões
EXTENSOES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
else:
    arquivos = sorted([p for p in PASTA_ENTRADA.iterdir() if p.suffix.lower() in EXTENSOES])
    escala   = REF_M / REF_PX
    print(f'📂 Entrada     : {PASTA_ENTRADA}')
    print(f'📁 Saída       : {PASTA_SAIDA}')
    print(f'📐 Calibração  : {REF_PX:.0f} px = {REF_M} m  →  escala = {escala:.5f} m/px')
    print(f'⚙️  SLIC        : n_segments={N_SEGMENTS}, compactness={COMPACTNESS}, sigma={SIGMA}')
    print(f'⚙️  Reparo (demo): y=[{REPARO_Y0}–{REPARO_Y1}], x=[{REPARO_X0}–{REPARO_X1}]')
    print(f'🖼️  {len(arquivos)} imagem(ns) encontrada(s)')

---
## Seção 3 — Pipeline SLIC + RAG + cálculo de área

### Fluxo completo

```
Imagem colorida (BGR)
        │
        ▼
  cvtColor → RGB          ← SLIC e scikit-image operam em RGB, não BGR
        │
        ▼
  SLIC (n_segments, compactness, sigma)
        │  → mapa de rótulos: cada pixel recebe ID do seu superpixel
        ▼
  RAG (rag_mean_color, mode='similarity')
        │  → grafo: nós = superpixels, arestas = similaridade de cor
        ▼
  Definir máscara do reparo ──────────────────────────────────────────┐
    (opções em ordem crescente de qualidade):                         │
    a) Retângulo proporcional [DEMO — este notebook]                  │
    b) Limiar de cor (HSV ou LAB) nos superpixels                     │
    c) Máscara PNG exportada do Label Studio                          │
    d) Clique interativo do usuário                                   │
    e) Modelo U-Net / SAM ← módulo seguinte                          │
        │  ←──────────────────────────────────────────────────────────┘
        ▼
  Contagem de pixels da máscara
        │
        ▼
  Área [px²] × escala² [m²/px²] = Área [m²]
        │
        ▼
  Salvar: superpixels PNG + máscara + comparativo + CSV
```

### Por que RAG se não usamos o merge aqui?

O RAG é construído mesmo na versão de demonstração porque:
1. Permite inspecionar quantos superpixels **caem dentro da máscara** — útil para validar a granularidade do SLIC
2. É a base para o próximo passo: mesclar superpixels por threshold de similaridade (`graph.cut_threshold`) para segmentação automática sem máscara manual
3. Exibir os pesos das arestas adjacentes à borda do reparo revela se o material do reparo é visualmente distinto o suficiente para segmentação automática

In [ ]:
def processar_imagem_regioes(caminho, ref_m, ref_px,
                              n_segments=500, compactness=10, sigma=1,
                              reparo_y=(0.30, 0.70), reparo_x=(0.20, 0.80),
                              largura_max=1200):
    """
    Aplica SLIC + RAG, define máscara de reparo e calcula área calibrada.
    Retorna (img_rgb, segments, rag, mascara, stats_dict).
    """
    img_bgr = cv2.imread(str(caminho))
    assert img_bgr is not None, f'Não foi possível abrir: {caminho.name}'

    if largura_max > 0 and img_bgr.shape[1] > largura_max:
        r       = largura_max / img_bgr.shape[1]
        img_bgr = cv2.resize(img_bgr, (largura_max, int(img_bgr.shape[0] * r)),
                             interpolation=cv2.INTER_AREA)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W    = img_rgb.shape[:2]

    # ── SLIC
    segments = slic(img_rgb, n_segments=n_segments,
                    compactness=compactness, sigma=sigma, start_label=1)
    n_sp_real = segments.max()   # número real de superpixels gerados

    # ── RAG
    rag = graph.rag_mean_color(img_rgb, segments, mode='similarity')

    # ── Máscara do reparo (retângulo proporcional — DEMO)
    y0 = int(H * reparo_y[0]);  y1 = int(H * reparo_y[1])
    x0 = int(W * reparo_x[0]);  x1 = int(W * reparo_x[1])
    mascara = np.zeros((H, W), dtype=np.uint8)
    mascara[y0:y1, x0:x1] = 255

    # ── Superpixels que intersectam a máscara
    ids_reparo = np.unique(segments[mascara > 0])

    # ── Calibração e área
    escala    = ref_m / ref_px           # m/px
    area_px   = int(mascara.sum() // 255)
    area_m2   = area_px * (escala ** 2)

    stats = {
        'arquivo'         : caminho.name,
        'largura_px'      : W,
        'altura_px'       : H,
        'n_superpixels'   : int(n_sp_real),
        'sp_no_reparo'    : len(ids_reparo),
        'area_reparo_px2' : area_px,
        'escala_m_px'     : round(escala, 6),
        'area_reparo_m2'  : round(area_m2, 3),
        'ref_m'           : ref_m,
        'ref_px'          : ref_px,
    }

    return img_rgb, img_bgr, segments, rag, mascara, stats


print('✅ Função do pipeline definida.')

---
## Seção 4 — Processamento em lote

Para cada imagem são salvos 4 arquivos:

```
fotos_obra_regioes/
  nome_foto_superpixels.png   ← superpixels SLIC sobre a imagem original
  nome_foto_mascara.png       ← máscara binária da área de reparo
  nome_foto_overlay.png       ← reparo destacado em vermelho translúcido
  nome_foto_comparativo.png   ← painel 4 vistas
  metricas_regioes.csv        ← área em m² e estatísticas de superpixels
```

In [ ]:
todas_stats = []
erros       = []

for arq in tqdm(arquivos, desc='Processando regiões'):
    try:
        img_rgb, img_bgr, segments, rag, mascara, stats = processar_imagem_regioes(
            arq,
            ref_m       = REF_M,
            ref_px      = REF_PX,
            n_segments  = N_SEGMENTS,
            compactness = COMPACTNESS,
            sigma       = SIGMA,
            reparo_y    = (REPARO_Y0, REPARO_Y1),
            reparo_x    = (REPARO_X0, REPARO_X1),
            largura_max = LARGURA_MAX,
        )

        base = arq.stem

        # ── Superpixels PNG
        sp_img = (mark_boundaries(img_rgb, segments, color=(1, 0.85, 0)) * 255).astype(np.uint8)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_superpixels.png'),
                    cv2.cvtColor(sp_img, cv2.COLOR_RGB2BGR))

        # ── Máscara PNG
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_mascara.png'), mascara)

        # ── Overlay vermelho translúcido
        overlay = img_bgr.copy()
        px  = img_bgr[mascara > 0]
        cor = np.full_like(px, [0, 0, 200])
        overlay[mascara > 0] = cv2.addWeighted(px, 0.45, cor, 0.55, 0)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_overlay.png'), overlay)

        # ── Painel comparativo 2×2
        H, W = img_bgr.shape[:2]
        def bgr3(m): return cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
        painel_top = np.hstack([img_bgr, cv2.cvtColor(sp_img, cv2.COLOR_RGB2BGR)])
        painel_bot = np.hstack([bgr3(mascara), overlay])
        painel     = np.vstack([painel_top, painel_bot])

        cv2.rectangle(painel, (10, 10), (750, 62), (255, 255, 255), -1)
        cv2.putText(painel,
                    f'SP={stats["n_superpixels"]}  SP_reparo={stats["sp_no_reparo"]}  '
                    f'Area={stats["area_reparo_m2"]:.2f} m2  escala={stats["escala_m_px"]:.5f} m/px',
                    (18, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (20, 20, 20), 2, cv2.LINE_AA)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_comparativo.png'), painel)

        todas_stats.append(stats)

    except Exception as e:
        erros.append((arq.name, str(e)))

# ── CSV
df = pd.DataFrame(todas_stats)
caminho_csv = PASTA_SAIDA / 'metricas_regioes.csv'
df.to_csv(str(caminho_csv), index=False)

print(f'\n✅ {len(todas_stats)} imagem(ns) processada(s)')
if len(df) > 0:
    print(f'   Área média  : {df["area_reparo_m2"].mean():.2f} m²')
    print(f'   Área máxima : {df["area_reparo_m2"].max():.2f} m²  ({df.loc[df["area_reparo_m2"].idxmax(), "arquivo"]})')
print(f'   Relatório   : {caminho_csv}')
if erros:
    print('\n⚠️  Erros:')
    for n, m in erros: print(f'   {n}: {m}')

---
## Seção 5 — Visualização, superpixels e métricas

### 5a. Painel completo por imagem

| Quadrante | O que analisar |
|-----------|----------------|
| **Original** | Referência — verifique se a faixa de escala está visível |
| **Superpixels** | Bordas amarelas devem se alinhar com mudanças de textura/cor. Se estiverem atravessando o reparo, reduza `compactness` |
| **Máscara** | Deve cobrir apenas a área do reparo — ajuste `REPARO_Y` e `REPARO_X` |
| **Overlay** | Verificação visual final — a área vermelha condiz com o reparo real? |

In [ ]:
MAX_EXIBIR = 4

for arq in arquivos[:MAX_EXIBIR]:
    img_rgb, img_bgr, segments, rag, mascara, stats = processar_imagem_regioes(
        arq, REF_M, REF_PX,
        N_SEGMENTS, COMPACTNESS, SIGMA,
        (REPARO_Y0, REPARO_Y1), (REPARO_X0, REPARO_X1),
        LARGURA_MAX,
    )

    sp_img  = mark_boundaries(img_rgb, segments, color=(1, 0.85, 0))
    overlay = img_bgr.copy()
    px  = img_bgr[mascara > 0]
    cor = np.full_like(px, [0, 0, 200])
    overlay[mascara > 0] = cv2.addWeighted(px, 0.45, cor, 0.55, 0)

    fig, eixos = plt.subplots(1, 4, figsize=(22, 5))
    dados = [
        (img_rgb,  'Original', None),
        (sp_img,   f'Superpixels SLIC\n({stats["n_superpixels"]} regiões)', None),
        (mascara,  f'Máscara do reparo\n({stats["sp_no_reparo"]} superpixels)', 'gray'),
        (cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
                   f'Overlay\nÁrea ≈ {stats["area_reparo_m2"]:.2f} m²', None),
    ]
    for ax, (im, t, cmap) in zip(eixos, dados):
        ax.imshow(im, cmap=cmap) if cmap else ax.imshow(im)
        ax.set_title(t, fontsize=9)
        ax.axis('off')

    plt.suptitle(
        f'{arq.name}   '
        f'Área = {stats["area_reparo_m2"]:.2f} m²   '
        f'Escala = {stats["escala_m_px"]:.5f} m/px',
        fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 5b. Distribuição de cores dos superpixels dentro × fora do reparo

Este gráfico responde à pergunta: **o reparo é visualmente distinto do asfalto original?**

Se as duas distribuições forem bem separadas, é possível segmentar automaticamente o reparo por cor (sem máscara manual), usando um limiar de brilho médio dos superpixels.

In [ ]:
if arquivos:
    img_rgb, img_bgr, segments, rag, mascara, stats = processar_imagem_regioes(
        arquivos[0], REF_M, REF_PX,
        N_SEGMENTS, COMPACTNESS, SIGMA,
        (REPARO_Y0, REPARO_Y1), (REPARO_X0, REPARO_X1),
        LARGURA_MAX,
    )

    gray = rgb2gray(img_rgb)

    brilhos_reparo = []
    brilhos_fundo  = []

    for sp_id in np.unique(segments):
        px_mask = segments == sp_id
        media = gray[px_mask].mean()
        if mascara[px_mask].mean() > 127:
            brilhos_reparo.append(media)
        else:
            brilhos_fundo.append(media)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.hist(brilhos_fundo,  bins=40, alpha=0.7, color='steelblue', label=f'Fundo ({len(brilhos_fundo)} SPs)')
    ax.hist(brilhos_reparo, bins=20, alpha=0.7, color='tomato',    label=f'Reparo ({len(brilhos_reparo)} SPs)')
    ax.set_xlabel('Brilho médio do superpixel (0–1)', fontsize=10)
    ax.set_ylabel('Número de superpixels', fontsize=10)
    ax.set_title(
        f'Distribuição de brilho — superpixels dentro × fora do reparo\n{arquivos[0].name}',
        fontsize=10)
    ax.legend()
    plt.tight_layout()
    plt.show()

    sep = abs(np.mean(brilhos_reparo) - np.mean(brilhos_fundo))
    print(f'Brilho médio — Reparo : {np.mean(brilhos_reparo):.3f}')
    print(f'Brilho médio — Fundo  : {np.mean(brilhos_fundo):.3f}')
    print(f'Separação              : {sep:.3f}')
    if sep > 0.08:
        print('✅ Separação suficiente → segmentação automática por cor é viável.')
    else:
        print('⚠️  Separação pequena → reparo e fundo similares → use máscara manual ou modelo.')

### 5c. Relatório de áreas — ranking por m²

In [ ]:
df_exib = df[['arquivo', 'n_superpixels', 'sp_no_reparo',
              'area_reparo_px2', 'area_reparo_m2', 'escala_m_px']]\
            .sort_values('area_reparo_m2', ascending=False).reset_index(drop=True)
df_exib.index += 1

def colorir_area(val):
    if   val > 20 : return 'background-color: #ffcccc'   # vermelho — grande
    elif val > 5  : return 'background-color: #fff3cd'   # amarelo — médio
    else          : return 'background-color: #d4edda'   # verde — pequeno

display(df_exib.style.applymap(colorir_area, subset=['area_reparo_m2']))

print('\n🔴 Vermelho : > 20 m² — reparo extenso, avaliar custo de recomposição')
print('🟡 Amarelo  : 5–20 m² — reparo médio')
print('🟢 Verde    : < 5 m²  — remendo pontual')
if len(df) > 0:
    print(f'\nÁrea total estimada : {df["area_reparo_m2"].sum():.2f} m²')

---
## Seção 6 — Calibração: grade de parâmetros SLIC

### O que calibrar e por quê

| Parâmetro | Efeito visual | Quando ajustar |
|-----------|--------------|----------------|
| `n_segments` | Mais superpixels = bordas mais precisas, porém mais fragmentação | Aumentar para reparos com bordas irregulares |
| `compactness` | Maior = superpixels mais quadrados (ignoram textura) | Reduzir quando a textura do reparo é distintiva |
| `sigma` | Maior = mais suavização antes do SLIC | Aumentar para imagens muito ruidosas |

A grade abaixo varia `n_segments` (linhas) × `compactness` (colunas) para a primeira imagem.

In [ ]:
if arquivos:
    img_ref_bgr = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img_ref_bgr.shape[1] > LARGURA_MAX:
        r           = LARGURA_MAX / img_ref_bgr.shape[1]
        img_ref_bgr = cv2.resize(img_ref_bgr, (LARGURA_MAX, int(img_ref_bgr.shape[0]*r)),
                                  interpolation=cv2.INTER_AREA)
    img_ref_rgb = cv2.cvtColor(img_ref_bgr, cv2.COLOR_BGR2RGB)

    n_segs      = [200, 400, 600]
    compacts    = [5, 10, 20]

    fig, eixos = plt.subplots(len(n_segs), len(compacts) + 1,
                               figsize=(5.5 * (len(compacts)+1), 5 * len(n_segs)))

    for i, ns in enumerate(n_segs):
        # Coluna 0 = original
        eixos[i][0].imshow(img_ref_rgb)
        eixos[i][0].set_ylabel(f'n_segments={ns}', fontsize=10, fontweight='bold')
        eixos[i][0].set_title('Original' if i == 0 else '', fontsize=9)
        eixos[i][0].axis('off')

        for j, cp in enumerate(compacts):
            segs = slic(img_ref_rgb, n_segments=ns, compactness=cp,
                        sigma=SIGMA, start_label=1)
            vis  = mark_boundaries(img_ref_rgb, segs, color=(1, 0.85, 0))
            eixos[i][j+1].imshow(vis)
            eixos[i][j+1].set_title(f'compactness={cp}\n{segs.max()} SPs reais' if i == 0
                                     else f'{segs.max()} SPs', fontsize=9)
            eixos[i][j+1].axis('off')

    plt.suptitle(f'Grade SLIC — {arquivos[0].name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('👆 Escolha a combinação onde as bordas dos superpixels se alinham')
    print('   com a fronteira entre reparo e asfalto original.')
    print('   Atualize N_SEGMENTS e COMPACTNESS na Seção 2 e reprocesse.')

---
## Próximos passos

### Evoluindo a máscara do reparo

O retângulo proporcional deste notebook é um **placeholder de demonstração**. O caminho de evolução:

```
[DEMO — este notebook]
Retângulo fixo (REPARO_Y, REPARO_X)
        │
        ▼
[Próximo passo — manual]
Limiar de brilho nos superpixels
→ if brilho_médio_sp > threshold: sp é reparo
→ funciona se separação (Seção 5b) for > 0.08
        │
        ▼
[Próximo passo — semi-auto]
Máscara exportada do Label Studio
→ substituir o bloco de máscara pela leitura do PNG
→ mesma lógica de calibração e área
        │
        ▼
[Módulo seguinte — automático]
U-Net ou SAM para segmentação sem interação
→ saída é máscara binária → área em m² pelo mesmo pipeline
```

### Integração com BIM

A área calculada em m² pode ser registrada diretamente como atributo IFC:

```
IfcSlab (pista) → Pset_SAC_Inspecao
    └── area_reparo_m2   : 12.45
    └── data_inspecao    : 2025-08-15
    └── metodo           : SLIC_RAG_calibrado
    └── escala_m_px      : 0.02000
```

### Referências

- Achanta, R. et al. (2012). *SLIC Superpixels Compared to State-of-the-Art Superpixel Methods*. IEEE TPAMI, 34(11).
- Felzenszwalb, P. & Huttenlocher, D. (2004). *Efficient Graph-Based Image Segmentation*. IJCV, 59(2).
- scikit-image docs: [`slic`](https://scikit-image.org/docs/stable/api/skimage.segmentation.html#skimage.segmentation.slic), [`rag_mean_color`](https://scikit-image.org/docs/stable/api/skimage.graph.html)
- DNIT (2006). *Manual de Restauração de Pavimentos Asfálticos* — critérios de medição de áreas de reparo
- ABNT NBR 7584:2012 — Concreto endurecido: avaliação da dureza superficial pelo esclerômetro de reflexão
